# Stage 8. Backbone Experiment

Notebook ini membandingkan tiga backbone deep embedding (ResNet18-CSA baseline, DINOv2-S self-supervised ViT, SigLIP-B vision-language ViT) pada situs nail, memakai threshold valid_pixel_mask baru (high=220) berdasarkan analisis empiris notebooks/analysis_valid_pixel.ipynb. Tujuannya mencari backbone yang menghasilkan representasi visual terbaik untuk skrining anemia dari citra kuku, terutama mengingat performa baseline saat ini (AUC ~0.72, accuracy ~0.64) masih dekat baseline trivial. Fitur handcrafted pada situs nail tidak dipakai model produksi (path_b_deep), sehingga eksperimen fokus pada Path B saja. Full Fusion tetap diuji sebagai pembanding untuk mengecek apakah backbone baru membuat handcrafted features menjadi berguna.

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import json

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score

from configs import paths
from src.common import features, manifest as manifest_utils, train

output_dir = paths.outputs_dir("nail")
artifact_dir = paths.artifacts_dir("nail")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)

PIXEL_HIGH = 220

print("sampel", len(manifest))
print("device", "cuda" if torch.cuda.is_available() else "cpu")
print("pixel_high", PIXEL_HIGH)

sampel 803
device cuda
pixel_high 220


## Re-extract Handcrafted Features (Threshold Baru)

Fitur handcrafted diekstrak ulang dengan pixel_high=220 agar konsisten dengan threshold baru. Meskipun model produksi nail saat ini tidak memakai handcrafted features, fitur ini tetap dibutuhkan untuk eksperimen Full Fusion sebagai pembanding.

In [2]:
handcrafted = features.extract_handcrafted_features(manifest, pixel_high=PIXEL_HIGH)
handcrafted.to_csv(output_dir / "handcrafted_features_t220.csv", index=False)
print("handcrafted features", handcrafted.shape)

handcrafted features (803, 28)


## Extract Deep Embeddings per Backbone

Tiga backbone diekstrak secara berurutan. Setiap backbone menghasilkan vektor 256-D per citra lewat layer proyeksi di EmbeddingBackbone. DINOv2 dan SigLIP memakai ukuran input yang berbeda (518 dan 224) sesuai spesifikasi pretraining masing-masing.

In [3]:
backbone_resnet = features.EmbeddingBackbone(backbone_name="resnet18", seed=42)
embeddings_resnet, uids_resnet = features.extract_deep_embeddings(
    manifest, model=backbone_resnet, size=backbone_resnet.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_resnet18_t220.npy", embeddings_resnet)
pd.DataFrame({"uid": uids_resnet}).to_csv(output_dir / "deep_embeddings_resnet18_t220_uids.csv", index=False)
print("resnet18", embeddings_resnet.shape)

resnet18 (803, 256)


In [4]:
backbone_dinov2 = features.EmbeddingBackbone(backbone_name="dinov2_small", seed=42)
embeddings_dinov2, uids_dinov2 = features.extract_deep_embeddings(
    manifest, model=backbone_dinov2, size=backbone_dinov2.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_dinov2s_t220.npy", embeddings_dinov2)
pd.DataFrame({"uid": uids_dinov2}).to_csv(output_dir / "deep_embeddings_dinov2s_t220_uids.csv", index=False)
print("dinov2_small", embeddings_dinov2.shape)

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

dinov2_small (803, 256)


In [5]:
backbone_siglip = features.EmbeddingBackbone(backbone_name="siglip_base_patch16_224", seed=42)
embeddings_siglip, uids_siglip = features.extract_deep_embeddings(
    manifest, model=backbone_siglip, size=backbone_siglip.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_siglip_t220.npy", embeddings_siglip)
pd.DataFrame({"uid": uids_siglip}).to_csv(output_dir / "deep_embeddings_siglip_t220_uids.csv", index=False)
print("siglip_base", embeddings_siglip.shape)

model.safetensors: reconstructing file:   0%|          |  0.00B /  372MB            

model.safetensors: downloading bytes:           |  0.00B            

siglip_base (803, 256)


## Backbone Comparison: Path B (Deep Only)

Setiap backbone diuji dengan konfigurasi Path B (deep embedding saja, tanpa handcrafted) memakai run_kfold dengan protokol identik (5-fold, 60 epoch). Metrik utama untuk nail adalah AUC dan F1 karena dataset timpang (32.6% anemic), bukan accuracy.

In [6]:
def _severity_kappa(oof):
    severity_valid = oof["severity_true"] >= 0
    if not severity_valid.any():
        return float("nan")
    return cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])


backbone_configs = {
    "resnet18": (embeddings_resnet, uids_resnet),
    "dinov2_small": (embeddings_dinov2, uids_dinov2),
    "siglip_base": (embeddings_siglip, uids_siglip),
}

path_b_results = {}
for name, (embeddings, uids) in backbone_configs.items():
    result = train.run_kfold(
        manifest, handcrafted, embeddings, uids,
        n_splits=5, epochs=60, use_handcrafted=False, use_deep=True,
    )
    path_b_results[name] = result
    fm = result["fold_metrics"]
    print(
        f"{name:20s}",
        f"MAE {fm['mae'].mean():.3f}",
        f"acc {fm['accuracy'].mean():.3f}",
        f"AUC {fm['auc'].mean():.3f}",
        f"F1 {fm['f1'].mean():.3f}",
        f"bal_acc {fm['balanced_accuracy'].mean():.3f}",
    )

resnet18             MAE 0.752 acc 0.624 AUC 0.701 F1 0.611 bal_acc 0.695


dinov2_small         MAE 0.854 acc 0.609 AUC 0.713 F1 0.568 bal_acc 0.653


siglip_base          MAE 0.749 acc 0.645 AUC 0.728 F1 0.615 bal_acc 0.702


## Backbone Comparison: Full Fusion

Full Fusion (deep embedding + 27 handcrafted features + demografi + site token) diuji untuk mengecek apakah backbone baru membuat handcrafted features menjadi berguna pada nail, yang sebelumnya tidak berkontribusi dengan ResNet18.

In [7]:
fusion_results = {}
for name, (embeddings, uids) in backbone_configs.items():
    result = train.run_kfold(
        manifest, handcrafted, embeddings, uids,
        n_splits=5, epochs=60, use_handcrafted=True, use_deep=True,
    )
    fusion_results[name] = result
    fm = result["fold_metrics"]
    print(
        f"{name:20s}",
        f"MAE {fm['mae'].mean():.3f}",
        f"acc {fm['accuracy'].mean():.3f}",
        f"AUC {fm['auc'].mean():.3f}",
        f"F1 {fm['f1'].mean():.3f}",
        f"bal_acc {fm['balanced_accuracy'].mean():.3f}",
    )

resnet18             MAE 0.759 acc 0.643 AUC 0.713 F1 0.613 bal_acc 0.700


dinov2_small         MAE 0.900 acc 0.648 AUC 0.713 F1 0.570 bal_acc 0.667


siglip_base          MAE 0.763 acc 0.643 AUC 0.727 F1 0.608 bal_acc 0.694


## Consolidated Comparison

Semua konfigurasi digabung ke satu tabel yang memakai AUC sebagai metrik utama (karena dataset nail timpang), dilengkapi F1, balanced accuracy, MAE, dan severity kappa.

In [8]:
summary_rows = []
for config_type, results_dict in [("path_b_deep", path_b_results), ("full_fusion", fusion_results)]:
    for backbone_name, result in results_dict.items():
        fm = result["fold_metrics"]
        summary_rows.append({
            "backbone": backbone_name,
            "config": config_type,
            "mae": fm["mae"].mean(),
            "accuracy": fm["accuracy"].mean(),
            "auc": fm["auc"].mean(),
            "f1": fm["f1"].mean(),
            "balanced_accuracy": fm["balanced_accuracy"].mean(),
            "severity_kappa": _severity_kappa(result["oof"]),
        })

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(output_dir / "backbone_experiment_comparison.csv", index=False)
print(summary_table.sort_values("auc", ascending=False).round(4).to_string(index=False))

    backbone      config    mae  accuracy    auc     f1  balanced_accuracy  severity_kappa
 siglip_base path_b_deep 0.7492    0.6450 0.7281 0.6150             0.7021          0.0862
 siglip_base full_fusion 0.7625    0.6425 0.7268 0.6076             0.6943          0.1504
dinov2_small path_b_deep 0.8543    0.6092 0.7134 0.5683             0.6531          0.1547
    resnet18 full_fusion 0.7585    0.6425 0.7134 0.6127             0.7003          0.0485
dinov2_small full_fusion 0.8999    0.6476 0.7133 0.5696             0.6666          0.1384
    resnet18 path_b_deep 0.7518    0.6240 0.7015 0.6108             0.6954          0.0033


## Hyperparameter Search on Best Backbone

Backbone dengan AUC tertinggi di tabel di atas dioptimasi lewat Optuna. Objective yang diminimalkan adalah 1 dikurangi AUC rata-rata, konsisten dengan pencarian hyperparameter pada notebook 05 nail.

In [9]:
import optuna

best_backbone_row = summary_table.sort_values("auc", ascending=False).iloc[0]
best_backbone_name = best_backbone_row["backbone"]
best_config_type = best_backbone_row["config"]
best_embeddings, best_uids = backbone_configs[best_backbone_name]
use_handcrafted_best = best_config_type == "full_fusion"

print(f"optimasi backbone: {best_backbone_name}, config: {best_config_type}")


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 30, 120)
    weight_classification = trial.suggest_float("weight_classification", 0.5, 3.0)
    weight_severity = trial.suggest_float("weight_severity", 0.1, 1.0)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    attention_dim = trial.suggest_categorical("attention_dim", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    result = train.run_kfold(
        manifest, handcrafted, best_embeddings, best_uids,
        n_splits=5, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
        loss_weights=(1.0, weight_classification, weight_severity),
        trunk_dim=trunk_dim, attention_dim=attention_dim, dropout=dropout, focal_gamma=focal_gamma,
        use_handcrafted=use_handcrafted_best, use_deep=True,
    )
    auc_mean = float(result["fold_metrics"]["auc"].mean())
    trial.set_user_attr("auc_mean", auc_mean)
    trial.set_user_attr("f1_mean", float(result["fold_metrics"]["f1"].mean()))
    trial.set_user_attr("mae_mean", float(result["fold_metrics"]["mae"].mean()))
    return 1.0 - auc_mean


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("best AUC", round(study.best_trial.user_attrs["auc_mean"], 4))
print("best F1", round(study.best_trial.user_attrs["f1_mean"], 4))
print("best MAE", round(study.best_trial.user_attrs["mae_mean"], 4))
print("best params", study.best_params)

[I 2026-09-30 13:23:42,017] A new study created in memory with name: no-name-906f161a-83da-448c-8eb3-76ddfb9b8baa


optimasi backbone: siglip_base, config: path_b_deep


[I 2026-09-30 13:23:46,763] Trial 0 finished with value: 0.3815523613385826 and parameters: {'learning_rate': 0.00013963963500413716, 'epochs': 87, 'weight_classification': 2.978418251179362, 'weight_severity': 0.8587222851014482, 'focal_gamma': 0.8614791849461474, 'dropout': 0.22087864981536576, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 128}. Best is trial 0 with value: 0.3815523613385826.


[I 2026-09-30 13:23:51,596] Trial 1 finished with value: 0.28008149025978357 and parameters: {'learning_rate': 0.00017848145069117, 'epochs': 88, 'weight_classification': 2.1933862582748267, 'weight_severity': 0.6868170883340277, 'focal_gamma': 1.4592183853212901, 'dropout': 0.27767059264269683, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 128}. Best is trial 1 with value: 0.28008149025978357.


[I 2026-09-30 13:24:10,129] Trial 2 finished with value: 0.27405522291708884 and parameters: {'learning_rate': 0.007436181519010792, 'epochs': 93, 'weight_classification': 1.7549343714692254, 'weight_severity': 0.4750210344908984, 'focal_gamma': 1.3618747269319917, 'dropout': 0.19797389750433278, 'trunk_dim': 256, 'attention_dim': 64, 'batch_size': 32}. Best is trial 2 with value: 0.27405522291708884.


...

[I 2026-09-30 13:31:17,540] Trial 46 finished with value: 0.2769392834932908 and parameters: {'learning_rate': 0.0049179662411580035, 'epochs': 73, 'weight_classification': 1.1745854359664578, 'weight_severity': 0.3754386824013002, 'focal_gamma': 2.0040468997650507, 'dropout': 0.15053570512222056, 'trunk_dim': 64, 'attention_dim': 64, 'batch_size': 64}. Best is trial 18 with value: 0.2659713437755673.


[I 2026-09-30 13:31:28,720] Trial 47 finished with value: 0.2732384216415721 and parameters: {'learning_rate': 0.001590631498516055, 'epochs': 57, 'weight_classification': 1.3758205833430164, 'weight_severity': 0.9077126596348261, 'focal_gamma': 2.7970567922933363, 'dropout': 0.12445156332995824, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 32}. Best is trial 18 with value: 0.2659713437755673.


[I 2026-09-30 13:31:35,679] Trial 48 finished with value: 0.2726953998245325 and parameters: {'learning_rate': 0.0003871802597432456, 'epochs': 35, 'weight_classification': 1.018489175782478, 'weight_severity': 0.1681881784084392, 'focal_gamma': 1.5408167821587175, 'dropout': 0.16558151091182421, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 32}. Best is trial 18 with value: 0.2659713437755673.


[I 2026-09-30 13:31:52,655] Trial 49 finished with value: 0.27571964199322685 and parameters: {'learning_rate': 0.0009801532745979129, 'epochs': 90, 'weight_classification': 2.001087105246029, 'weight_severity': 0.10060950705272592, 'focal_gamma': 1.3800627906917606, 'dropout': 0.22941797134880015, 'trunk_dim': 64, 'attention_dim': 32, 'batch_size': 32}. Best is trial 18 with value: 0.2659713437755673.


best AUC 0.734
best F1 0.6138
best MAE 0.7587
best params {'learning_rate': 0.00045493191019158286, 'epochs': 49, 'weight_classification': 0.9784813963476624, 'weight_severity': 0.24861345104264793, 'focal_gamma': 1.7445200635237699, 'dropout': 0.22807753401527125, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 32}


## Retrain with Best Hyperparameters

Backbone pemenang dilatih ulang memakai hyperparameter terbaik dari Optuna, lalu checkpoint disimpan untuk deployment.

In [10]:
best_params = study.best_params
tuned_result = train.run_kfold(
    manifest, handcrafted, best_embeddings, best_uids,
    n_splits=5,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"],
    attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"],
    focal_gamma=best_params["focal_gamma"],
    use_handcrafted=use_handcrafted_best, use_deep=True,
)

tuned_fm = tuned_result["fold_metrics"]
print("tuned MAE", round(tuned_fm["mae"].mean(), 4))
print("tuned accuracy", round(tuned_fm["accuracy"].mean(), 4))
print("tuned AUC", round(tuned_fm["auc"].mean(), 4))
print("tuned F1", round(tuned_fm["f1"].mean(), 4))
print("tuned balanced_accuracy", round(tuned_fm["balanced_accuracy"].mean(), 4))
print("tuned severity_kappa", round(_severity_kappa(tuned_result["oof"]), 4))

tuned MAE 0.7569
tuned accuracy 0.64
tuned AUC 0.731
tuned F1 0.6099
tuned balanced_accuracy 0.6975
tuned severity_kappa 0.16


## Save Checkpoints and Results

In [11]:
tuned_result["oof"].to_csv(output_dir / f"multitask_oof_{best_backbone_name}_{best_config_type}_tuned.csv", index=False)

for fold_index, fold_model in enumerate(tuned_result["models"]):
    checkpoint_path = artifact_dir / f"multitask_{best_backbone_name}_{best_config_type}_tuned_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

with open(output_dir / "backbone_experiment_best.json", "w") as handle:
    json.dump({
        "backbone": best_backbone_name,
        "config": best_config_type,
        "use_handcrafted": use_handcrafted_best,
        "pixel_high": PIXEL_HIGH,
        "best_params": best_params,
        "auc": study.best_trial.user_attrs["auc_mean"],
        "f1": study.best_trial.user_attrs["f1_mean"],
        "mae": study.best_trial.user_attrs["mae_mean"],
    }, handle, indent=2)

print(f"checkpoint disimpan untuk {best_backbone_name} {best_config_type}")

checkpoint disimpan untuk siglip_base path_b_deep
